# MaizeScan — Training Notebook v1
## Binary CNN: Healthy vs. Diseased Maize Leaves (InceptionV3)

### Before you start
1. **Enable GPU:** Runtime → Change runtime type → Hardware accelerator: **T4 GPU**
2. **Kaggle token:** [kaggle.com → Settings → API → Create New Token](https://www.kaggle.com/settings/account) → download `kaggle.json`
3. **Google Drive:** Authorize when prompted — checkpoints and artifacts persist across sessions

### Dual-dataset training
This notebook merges two complementary Kaggle datasets into a unified binary class structure:

| Dataset | Kaggle slug | Format | Contribution |
|---------|-------------|--------|--------------|
| Maize in Field | `hamishcrazeai/maize-in-field-dataset` | CSV + field photos | Real-world field conditions |
| New Plant Diseases | `vipoooool/new-plant-diseases-dataset` | Folder per class (PlantVillage) | Controlled lab conditions |

Both are consolidated into `master_binary_dataset/Healthy/` and `master_binary_dataset/Diseased/`
before splitting and training (FIX-8).

### Architecture
- **Base**: InceptionV3 (ImageNet pretrained, 299×299 native — standardised to 224×224)
- **Head**: GAP → Dense(256, relu) → BatchNorm → Dropout(0.5) → Dense(1, sigmoid)
- **Phase 1**: Freeze base, train head only (25 epochs, LR=1e-3)
- **Phase 2**: Unfreeze top 93 layers (mixed7 onward), fine-tune (50 epochs, LR=1e-5 + linear warmup)
- **Export**: INT8 quantized TFLite — API expects `inceptionv3_best.tflite`

### Estimated runtime
| Step | Time |
|------|------|
| Environment setup | ~5 min |
| Dataset download (~3 GB total) | ~12 min |
| Master dataset construction | ~5 min |
| Phase 1 training | ~20 min |
| Phase 2 fine-tuning | ~40 min |
| Export + benchmark | ~5 min |
| **Total** | **~87 min** |

### Output artifacts
- `inceptionv3_best.tflite` — INT8 quantized model for the FastAPI backend
- `inceptionv3_meta.json` — metrics + version metadata read by `/health` and `/model/info`

Both are saved to Google Drive **and** downloaded to your machine at the end.

In [ ]:
# ── Cell 1: Environment Setup ─────────────────────────────────────────────────
import subprocess, sys, os, shutil, json, random, logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

# Install pinned TF version (matches production Dockerfile.api)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'tensorflow==2.15.0',
    'scikit-learn>=1.3',
    'kaggle>=1.6',
    'split-folders>=0.5',
    'pandas>=2.0',
    'pillow>=10.0',
    'matplotlib>=3.8',
    'seaborn>=0.13',
])

# Mount Google Drive for checkpoint + artifact persistence
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = Path('/content/drive/MyDrive/MaizeScan')
DRIVE_BASE.mkdir(parents=True, exist_ok=True)

# Clone repo so we can import model.* modules
REPO_URL = 'https://github.com/CBahtaria/maize-leaf-classifier.git'
REPO_DIR = Path('/content/maize-leaf-classifier')
if not REPO_DIR.exists():
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)])
else:
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'])

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

OUTPUT_DIR = DRIVE_BASE / 'training_output'
ARTIFACTS  = DRIVE_BASE / 'model_artifacts'
for p in [OUTPUT_DIR, ARTIFACTS]:
    p.mkdir(parents=True, exist_ok=True)

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f'TensorFlow {tf.__version__}  |  GPUs: {[g.name for g in gpus]}')
if not gpus:
    print('WARNING: No GPU — go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 2: Kaggle Credentials + Dual Dataset Download ────────────────────────
from google.colab import files as colab_files
import stat

KAGGLE_CACHE = DRIVE_BASE / 'kaggle.json'
KAGGLE_DIR   = Path('/root/.kaggle')
KAGGLE_DIR.mkdir(parents=True, exist_ok=True)

if KAGGLE_CACHE.exists():
    shutil.copy(KAGGLE_CACHE, KAGGLE_DIR / 'kaggle.json')
    print('Kaggle token loaded from Drive cache.')
else:
    print('Upload your kaggle.json now (file picker will appear):')
    uploaded = colab_files.upload()
    for fname in uploaded:
        src = Path('/content') / fname
        if not src.exists():
            src = Path(fname)
        shutil.copy(src, KAGGLE_DIR / 'kaggle.json')
        shutil.copy(src, KAGGLE_CACHE)
        print('Cached kaggle.json to Drive for future sessions.')

(KAGGLE_DIR / 'kaggle.json').chmod(stat.S_IRUSR | stat.S_IWUSR)

# Download both datasets
os.chdir('/content')
print('\nDownloading Maize in Field dataset (~500 MB)...')
subprocess.check_call(['kaggle', 'datasets', 'download',
    '-d', 'hamishcrazeai/maize-in-field-dataset'])

print('Downloading New Plant Diseases dataset (~2.5 GB)...')
subprocess.check_call(['kaggle', 'datasets', 'download',
    '-d', 'vipoooool/new-plant-diseases-dataset'])

print('Extracting...')
Path('/content/maize_dataset').mkdir(exist_ok=True)
Path('/content/new_plant_dataset').mkdir(exist_ok=True)
subprocess.check_call(['unzip', '-q', '-o',
    'maize-in-field-dataset.zip', '-d', '/content/maize_dataset/'])
subprocess.check_call(['unzip', '-q', '-o',
    'new-plant-diseases-dataset.zip', '-d', '/content/new_plant_dataset/'])
print('Extraction complete.')

In [ ]:
# ── Cell 3: Build Master Binary Dataset ───────────────────────────────────────
# Merges both sources into master_binary_dataset/Healthy/ + Diseased/
# model/preprocess.py::scan_dataset() reads this structure directly (FIX-8).
import pandas as pd

MASTER_DIR   = Path('/content/master_binary_dataset')
HEALTHY_DIR  = MASTER_DIR / 'Healthy'
DISEASED_DIR = MASTER_DIR / 'Diseased'
HEALTHY_DIR.mkdir(parents=True, exist_ok=True)
DISEASED_DIR.mkdir(parents=True, exist_ok=True)

_IMG_EXTS = {'.jpg', '.jpeg', '.png'}

def _copy_image(src: Path, dest_dir: Path, prefix: str) -> bool:
    if src.suffix.lower() in _IMG_EXTS and src.exists():
        shutil.copy(src, dest_dir / f'{prefix}{src.name}')
        return True
    return False

# ── PART A: CSV-based (Maize in Field) ────────────────────────────────────────
# NoFoliarSymptoms == 1 → no visible symptoms → Healthy  (label 0)
# NoFoliarSymptoms == 0 → symptoms present   → Diseased (label 1)
csv_path   = Path('/content/maize_dataset/Kaggle Dataset/Database.csv')
image_base = Path('/content/maize_dataset/Kaggle Dataset/leaf_images/')

df = pd.read_csv(csv_path)
df['binary_label']  = (df['NoFoliarSymptoms'] == 0).astype(int)
df['absolute_path'] = df['filePath'].apply(
    lambda x: image_base / os.path.basename(str(x)))

csv_h, csv_d = 0, 0
for _, row in df.iterrows():
    src  = Path(row['absolute_path'])
    dest = DISEASED_DIR if row['binary_label'] == 1 else HEALTHY_DIR
    if _copy_image(src, dest, 'csv_field_'):
        if row['binary_label'] == 1:
            csv_d += 1
        else:
            csv_h += 1

print(f'CSV dataset  → Healthy: {csv_h:,}   Diseased: {csv_d:,}')

# ── PART B: Folder-based (New Plant Diseases — Corn/maize classes only) ────────
# Locate the split root (zip may have nested folders)
np_candidates = list(Path('/content/new_plant_dataset').rglob('train'))
NP_SPLIT_BASE = np_candidates[0].parent if np_candidates else Path('/content/new_plant_dataset')

FOLDER_MAP = {
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot': (DISEASED_DIR, 'gls'),
    'Corn_(maize)___Common_rust_':                        (DISEASED_DIR, 'rust'),
    'Corn_(maize)___Northern_Leaf_Blight':                (DISEASED_DIR, 'blight'),
    'Corn_(maize)___healthy':                             (HEALTHY_DIR,  'healthy'),
}

np_h, np_d = 0, 0
for split in ['train', 'valid']:
    split_dir = NP_SPLIT_BASE / split
    if not split_dir.exists():
        continue
    for folder_name, (dest_dir, tag) in FOLDER_MAP.items():
        src_folder = split_dir / folder_name
        if not src_folder.exists():
            continue
        for img_path in src_folder.iterdir():
            prefix = f'np_{split}_{tag}_'
            if _copy_image(img_path, dest_dir, prefix):
                if dest_dir == DISEASED_DIR:
                    np_d += 1
                else:
                    np_h += 1

print(f'PlantVillage → Healthy: {np_h:,}   Diseased: {np_d:,}')

total_h = len(list(HEALTHY_DIR.iterdir()))
total_d = len(list(DISEASED_DIR.iterdir()))
print(f'\n── Unified Master Dataset ─────────────────')
print(f'  Healthy  (class 0): {total_h:>6,}')
print(f'  Diseased (class 1): {total_d:>6,}')
print(f'  Total             : {total_h + total_d:>6,}')

In [ ]:
# ── Cell 4: Scan, Split, Class Weights + Sample Grid ─────────────────────────
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from model.preprocess import scan_dataset, stratified_split, compute_class_weights

# binary_map() in preprocess.py is case-insensitive:
# 'Healthy' → 0, 'Diseased' → 1
paths, labels = scan_dataset(MASTER_DIR)
(train_p, train_l), (val_p, val_l), (test_p, test_l) = stratified_split(paths, labels)
class_weights = compute_class_weights(train_l)

print('── Dataset Split ──────────────────────────')
print(f'  Train : {len(train_p):>6,}  '
      f'(healthy={train_l.count(0):,}  diseased={train_l.count(1):,})')
print(f'  Val   : {len(val_p):>6,}  '
      f'(healthy={val_l.count(0):,}  diseased={val_l.count(1):,})')
print(f'  Test  : {len(test_p):>6,}  '
      f'(healthy={test_l.count(0):,}  diseased={test_l.count(1):,})')
print(f'\n  Class weights  →  healthy: {class_weights[0]:.3f}  '
      f'diseased: {class_weights[1]:.3f}')

# 5×2 sample grid
healthy_p  = [p for p, l in zip(paths, labels) if l == 0]
diseased_p = [p for p, l in zip(paths, labels) if l == 1]
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Sample Images — Healthy (top) vs Diseased (bottom)', fontsize=13)
for i, p in enumerate(random.sample(healthy_p, 5)):
    axes[0, i].imshow(Image.open(p).resize((224, 224)))
    axes[0, i].axis('off')
    axes[0, i].set_title('Healthy', fontsize=9, color='green')
for i, p in enumerate(random.sample(diseased_p, 5)):
    axes[1, i].imshow(Image.open(p).resize((224, 224)))
    axes[1, i].axis('off')
    axes[1, i].set_title('Diseased', fontsize=9, color='red')
plt.tight_layout()
plt.savefig(str(ARTIFACTS / 'sample_grid.png'), dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 5: Build Datasets + Train InceptionV3 ────────────────────────────────
# Expected runtime: ~60 min on T4 GPU
# Checkpoints saved to Drive after each phase — safe to disconnect and resume.
from model.preprocess import build_tf_dataset
from model.train import train as train_model

ARCH = 'inceptionv3'
# preprocess_fn is embedded as a Lambda layer inside the model (FIX-1).
# build_tf_dataset yields raw [0,255] images — the model normalises internally.

print('Building tf.data pipelines...')
train_ds = build_tf_dataset(train_p, train_l, augment=True)
val_ds   = build_tf_dataset(val_p,   val_l,   augment=False)
test_ds  = build_tf_dataset(test_p,  test_l,  augment=False)

# 200-image representative dataset for INT8 quantization calibration (FIX-3)
rep_sample = random.sample(train_p, min(200, len(train_p)))
rep_images = np.stack([
    np.array(Image.open(p).resize((224, 224)), dtype=np.float32)
    for p in rep_sample
])  # shape (200, 224, 224, 3) — raw [0,255], preprocess_fn is embedded in model
print(f'Calibration images: {rep_images.shape}  dtype={rep_images.dtype}')

print(f'\nStarting two-phase training — output: {OUTPUT_DIR}\n')
results = train_model(
    arch_name=ARCH,
    train_ds=train_ds,
    val_ds=val_ds,
    test_ds=test_ds,
    class_weights=class_weights,
    output_dir=str(OUTPUT_DIR),
    representative_images=rep_images,
)
print('Training complete.')
print(f"  Model  : {results['model_path']}")
print(f"  TFLite : {results['tflite_path']}")

In [ ]:
# ── Cell 6: Evaluation Metrics + Plots ────────────────────────────────────────
from IPython.display import Image as IPImage, display
from model.evaluate import generate_plots

m  = results['metrics']
ci = m.get('wilson_ci_95', {})

print('=' * 55)
print(f'  Architecture : InceptionV3 (binary classifier)')
print('=' * 55)
print(f"  Accuracy     : {m['accuracy']:.4f}")
print(f"  Precision    : {m['precision']:.4f}")
print(f"  Sensitivity  : {m['sensitivity']:.4f}  (Recall / TPR)")
print(f"  Specificity  : {m['specificity']:.4f}  (TNR)")
print(f"  F1-Score     : {m['f1']:.4f}")
print(f"  AUC-ROC      : {m['auc_roc']:.4f}")
print('=' * 55)

# Generate confusion matrix + ROC curve + training history
generate_plots(results, output_dir=str(ARTIFACTS), arch_name=ARCH)

for plot_name in [f'{ARCH}_confusion_matrix.png',
                  f'{ARCH}_roc_curve.png',
                  f'{ARCH}_training_history.png']:
    fpath = ARTIFACTS / plot_name
    if fpath.exists():
        print(f'\n── {plot_name} ──')
        display(IPImage(str(fpath), width=500))

In [ ]:
# ── Cell 7: Rename Artifacts + Benchmark TFLite ───────────────────────────────
# train() saves as inceptionv3_int8.tflite — rename to inceptionv3_best.tflite
# so the API's MODEL_PATH=model_artifacts/inceptionv3_best.tflite works out of box.
from model.evaluate import measure_inference_time_tflite

src_tflite = Path(results['tflite_path'])        # .../inceptionv3_int8.tflite
src_meta   = src_tflite.parent / 'inceptionv3_meta.json'

dst_tflite = ARTIFACTS / 'inceptionv3_best.tflite'
dst_meta   = ARTIFACTS / 'inceptionv3_meta.json'

shutil.copy(src_tflite, dst_tflite)
shutil.copy(src_meta,   dst_meta)

size_mb = dst_tflite.stat().st_size / 1e6
print(f'Artifacts copied to Drive:')
print(f'  {dst_tflite}  ({size_mb:.1f} MB)')
print(f'  {dst_meta}')

# Benchmark TFLite CPU inference (FIX-7: matches deployment, not academic .keras benchmark)
print('\nBenchmarking TFLite inference (CPU, 50 runs)...')
timing = measure_inference_time_tflite(str(dst_tflite), n_runs=50)
print(f"  Mean  : {timing['mean_ms']:.1f} ms")
print(f"  Median: {timing['median_ms']:.1f} ms")
print(f"  P95   : {timing['p95_ms']:.1f} ms")

In [ ]:
# ── Cell 8: Download Artifacts ────────────────────────────────────────────────
# Downloads both files to your local machine.
# After downloading:
#   mkdir -p model_artifacts
#   mv ~/Downloads/inceptionv3_best.tflite  model_artifacts/
#   mv ~/Downloads/inceptionv3_meta.json    model_artifacts/
#
# .env.example already has the correct defaults:
#   MODEL_PATH=model_artifacts/inceptionv3_best.tflite
#   MODEL_META_PATH=model_artifacts/inceptionv3_meta.json
#
# Deploy to VPS:
#   bash scripts/upload-model.sh deploy@<VPS_IP>
from google.colab import files as colab_files

print('Downloading inceptionv3_best.tflite...')
colab_files.download(str(dst_tflite))

print('Downloading inceptionv3_meta.json...')
colab_files.download(str(dst_meta))

print("""
\u256d\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u256e
\u2502  Training complete!                                        \u2502
\u2502                                                            \u2502
\u2502  Downloaded:                                               \u2502
\u2502    inceptionv3_best.tflite  (INT8 quantized)               \u2502
\u2502    inceptionv3_meta.json    (metrics + version info)       \u2502
\u2502                                                            \u2502
\u2502  Move both to model_artifacts/ then run:                   \u2502
\u2502    docker compose -f docker/docker-compose.dev.yml up      \u2502
\u2502    curl localhost:8000/health                               \u2502
\u2502                                                            \u2502
\u2502  Or deploy directly to VPS:                                \u2502
\u2502    bash scripts/upload-model.sh deploy@<your-server-ip>    \u2502
\u2570\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u256f
""")

## Research Inconsistencies Fixed in This Implementation

| Fix | Paper Section | Issue | Resolution |
|-----|--------------|-------|------------|
| FIX-1 | 3.5.3 | `/255` normalization applied to all architectures — wrong for MobileNetV2, Xception, InceptionV3 | Architecture-specific `preprocess_input` embedded as Lambda layer; model accepts raw `[0,255]` input (`model/architectures.py`) |
| FIX-2 | 3.6.1 | Deprecated `ImageDataGenerator` used for preprocessing | `tf.data.Dataset` pipeline in `model/preprocess.py` with `AUTOTUNE` prefetching |
| FIX-3 | 3.12.2 | Paper claims mobile deployment but never specifies TFLite conversion | Full INT8 post-training quantization in `model/export.py`; calibrated with 200 representative images |
| FIX-4 | 3.10.2 | Linear LR warmup described in paper but no Keras callback exists natively | `LinearWarmupCallback` in `model/callbacks.py`; uses `.assign()` for TF 2.x compatibility |
| FIX-5 | Table 3.7 | Hardcoded layer unfreeze index breaks across TF versions | Dynamic `len(base_model.layers) - fine_tune_n` in `model/architectures.py::unfreeze_top_n()` |
| FIX-6 | 3.7 | Class weight formula applied inconsistently | Inverse-frequency formula `w_c = N_total / (N_classes × N_c)` in `model/preprocess.py::compute_class_weights()` |
| FIX-7 | 3.12.2 | CPU benchmarks measured on full `.keras` model, not the deployed TFLite | `measure_inference_time_tflite()` in `model/evaluate.py` benchmarks the INT8 TFLite file |
| FIX-8 | 3.4 | Single-source dataset (PlantVillage only) lacks real-world field variation | Dual-dataset training: Maize in Field (CSV, field photos) + New Plant Diseases (PlantVillage lab) merged into unified binary structure |